In [1]:
import os
import time
import json

import openai
from openai import OpenAI

from elasticsearch import Elasticsearch

import spacy
import numpy as np

import pandas as pd
from tqdm.auto import tqdm

In [55]:
df_ground_truth_spacy = pd.read_csv('../data/ground_truth_NEW.csv')
ground_truth = df_ground_truth_spacy.to_dict(orient='records')

df_ground_truth_st = pd.read_csv('../data/ground_truth_NEW.csv')


In [3]:
ground_truth[0]

{'question': 'How do I sign up for an account on your website?',
 'document': 'doc_0_how_can_i_create_an_account_'}

In [4]:
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [5]:
es_client = Elasticsearch('http://localhost:9200')
es_client

<Elasticsearch(['http://localhost:9200'])>

In [ ]:
# Calculating for index generated using Spacy embeddings
index_name='documents_spacy'
try:
    result = es_client.count(index=index_name)
    print(f"ES Checking = Document count in {index_name}: {result['count']}")
except Exception as e:
    print(f"ES Checking = Error: {str(e)}")

ES Checking = Document count in documents: 79


In [6]:
# Calculating for index generated using SentenceTransformer embeddings
index_name='documents_st'
try:
    result = es_client.count(index=index_name)
    print(f"ES Checking = Document count in {index_name}: {result['count']}")
except Exception as e:
    print(f"ES Checking = Error: {str(e)}")

ES Checking = Document count in documents_st: 79


In [7]:
q = ground_truth[0]['question']
d = ground_truth[0]['document']
q, d

('How do I sign up for an account on your website?',
 'doc_0_how_can_i_create_an_account_')

In [9]:
def elastic_search_text(query, index_name="documents_spacy"):
    search_query = {
        "size": 10,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "answer"],
                        "type": "best_fields"
                    }
                },
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)
    return [hit["_source"] for hit in response["hits"]["hits"]]

In [10]:
# Calculating for index generated using SentenceTransformer embeddings
index_name='documents_st'

text_results_st = elastic_search_text(ground_truth[0]['question'], index_name=index_name)
text_results_st = [item['document_id'] for item in text_results_st]
text_results_st

['doc_0_how_can_i_create_an_account_',
 'doc_64_can_i_request_a_product_if_it_',
 'doc_16_can_i_order_without_creating_a',
 'doc_30_do_you_offer_installation_serv',
 'doc_62_can_i_order_a_product_if_it_is',
 'doc_37_can_i_request_an_invoice_for_m',
 'doc_2_how_can_i_track_my_order_',
 'doc_9_how_can_i_contact_customer_sup',
 'doc_5_how_long_does_shipping_take_',
 'doc_19_how_can_i_leave_a_product_revi']

In [11]:

# Calculating for index generated using SentenceTransformer embeddings
index_name='documents_spacy'

text_results_spacy = elastic_search_text(ground_truth[0]['question'], index_name=index_name)
text_results_spacy = [item['document_id'] for item in text_results_spacy]
text_results_spacy

['doc_0_how_can_i_create_an_account_',
 'doc_64_can_i_request_a_product_if_it_',
 'doc_16_can_i_order_without_creating_a',
 'doc_30_do_you_offer_installation_serv',
 'doc_62_can_i_order_a_product_if_it_is',
 'doc_37_can_i_request_an_invoice_for_m',
 'doc_2_how_can_i_track_my_order_',
 'doc_9_how_can_i_contact_customer_sup',
 'doc_5_how_long_does_shipping_take_',
 'doc_19_how_can_i_leave_a_product_revi']

In [ ]:
import spacy
import numpy as np
from sentence_transformers import SentenceTransformer

nlp = spacy.load('en_core_web_sm')

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
def get_vector(query, embedding_type='Spacy'):
    if embedding_type == 'Spacy':
        doc = nlp(query)
        tokens = [token.lemma_ for token in doc]
        text = ' '.join(tokens)
        doc_lemmatized = nlp(text)
        vector = np.mean([token.vector for token in doc_lemmatized], axis=0).tolist()
    elif embedding_type == 'SentenceTransformer':
        vector = model.encode(query).tolist()
    return vector

In [16]:
def elastic_search_knn(query,embedding_type='Spacy', index_name="documents_spacy", field='embedding'):
                
    vector = get_vector(query, embedding_type)

    search_body = {
        "knn": {
            "field": "embedding",
            "query_vector": vector,
            "k": 10,
            "num_candidates": 100
        },
        "size": 10,
        "_source": ['document_id', 'question', 'answer'],
    }

    es_results = es_client.search(index=index_name, body=search_body)

    return [hit["_source"] for hit in es_results["hits"]["hits"]]

In [17]:
embedding_type = 'SentenceTransformer'
index_name = 'documents_st'
knn_results_st = elastic_search_knn(ground_truth[0]['question'], embedding_type, index_name)
knn_results_st = [item['document_id'] for item in knn_results_st]
knn_results_st

['doc_0_how_can_i_create_an_account_',
 'doc_16_can_i_order_without_creating_a',
 'doc_15_do_you_have_a_loyalty_program_',
 'doc_64_can_i_request_a_product_if_it_',
 'doc_12_can_i_order_by_phone_',
 'doc_1_what_payment_methods_do_you_ac',
 'doc_28_what_should_i_do_if_my_discoun',
 'doc_35_can_i_request_a_product_demons',
 'doc_11_what_is_your_price_matching_po',
 'doc_2_how_can_i_track_my_order_']

In [20]:
embedding_type = 'Spacy'
index_name = 'documents_spacy'
knn_results_spacy = elastic_search_knn(ground_truth[0]['question'], embedding_type, index_name)
knn_results_spacy = [item['document_id'] for item in knn_results_spacy]
knn_results_spacy

['doc_70_can_i_request_a_product_that_i',
 'doc_40_can_i_request_a_product_that_i',
 'doc_23_can_i_order_a_product_that_is_',
 'doc_21_what_should_i_do_if_i_receive_',
 'doc_64_can_i_request_a_product_if_it_',
 'doc_69_can_i_return_a_product_if_it_w',
 'doc_7_what_should_i_do_if_my_package',
 'doc_67_can_i_request_a_product_that_i',
 'doc_76_can_i_request_a_product_if_it_',
 'doc_73_can_i_request_a_product_that_i']

In [21]:
ground_truth[:3]

[{'question': 'How do I sign up for an account on your website?',
  'document': 'doc_0_how_can_i_create_an_account_'},
 {'question': 'Where can I find the Sign Up button to make a new account?',
  'document': 'doc_0_how_can_i_create_an_account_'},
 {'question': 'What steps do I need to follow to register a new account?',
  'document': 'doc_0_how_can_i_create_an_account_'}]

In [56]:
truth_question = df_ground_truth_spacy.iloc[0]['question']
truth_document_id = df_ground_truth_spacy.iloc[0]['document']
truth_question, truth_document_id

('How do I sign up for an account on your website?',
 'doc_0_how_can_i_create_an_account_')

In [23]:
def hit_rate_one(original_id, search_results):
    return 1 if original_id in search_results else 0

In [24]:
def mrr_one(original_id, search_results):
    mrr = 0
    for position in range(len(search_results)):
        if search_results[position] == original_id:
            mrr += 1 / (position + 1)
    return mrr

In [ ]:
text_results_spacy = elastic_search_text(truth_question)
text_results_spacy = [item['document_id'] for item in text_results_spacy]
hit_rate_one(truth_document_id, text_results_spacy)
mrr_one(truth_document_id, text_results_spacy)
text_results_spacy

(['doc_0_how_can_i_create_an_account_',
  'doc_64_can_i_request_a_product_if_it_',
  'doc_16_can_i_order_without_creating_a',
  'doc_30_do_you_offer_installation_serv',
  'doc_62_can_i_order_a_product_if_it_is',
  'doc_37_can_i_request_an_invoice_for_m',
  'doc_2_how_can_i_track_my_order_',
  'doc_9_how_can_i_contact_customer_sup',
  'doc_5_how_long_does_shipping_take_',
  'doc_19_how_can_i_leave_a_product_revi'],
 1)

In [28]:
knn_results_st = elastic_search_knn(truth_question, embedding_type='SentenceTransformer', index_name='documents_st')

knn_results_st = [item['document_id'] for item in knn_results_st]
hit_rate_one(truth_document_id, knn_results_st)
mrr_one(truth_document_id, knn_results_st)
knn_results_st

['doc_0_how_can_i_create_an_account_',
 'doc_16_can_i_order_without_creating_a',
 'doc_15_do_you_have_a_loyalty_program_',
 'doc_64_can_i_request_a_product_if_it_',
 'doc_12_can_i_order_by_phone_',
 'doc_1_what_payment_methods_do_you_ac',
 'doc_28_what_should_i_do_if_my_discoun',
 'doc_35_can_i_request_a_product_demons',
 'doc_11_what_is_your_price_matching_po',
 'doc_2_how_can_i_track_my_order_']

In [29]:
knn_results_spacy = elastic_search_knn(truth_question)

knn_results_spacy = [item['document_id'] for item in knn_results_spacy]
hit_rate_one(truth_document_id, knn_results_spacy)
mrr_one(truth_document_id, knn_results_spacy)
knn_results_spacy

['doc_70_can_i_request_a_product_that_i',
 'doc_40_can_i_request_a_product_that_i',
 'doc_23_can_i_order_a_product_that_is_',
 'doc_21_what_should_i_do_if_i_receive_',
 'doc_64_can_i_request_a_product_if_it_',
 'doc_69_can_i_return_a_product_if_it_w',
 'doc_7_what_should_i_do_if_my_package',
 'doc_67_can_i_request_a_product_that_i',
 'doc_76_can_i_request_a_product_if_it_',
 'doc_73_can_i_request_a_product_that_i']

In [31]:
def elastic_search_knn_combined_style(query,embedding_type='Spacy', index_name="documents_spacy", field='embedding'):
    # Obtain the vector representation of the query
    vector = get_vector(query, embedding_type)

    # Construct the search query using a script score for cosine similarity
    search_query = {
        "size": 10,  # Number of results to return
        "query": {
            "bool": {
                "must": [
                    {
                        "script_score": {
                            "query": {
                                "match_all": {}  # Match all documents to apply custom scoring
                            },
                            "script": {
                                "source": """
                                    cosineSimilarity(params.query_vector, 'embedding') + 1
                                """,  # +1 to ensure the score is positive
                                "params": {
                                    "query_vector": vector
                                }
                            }
                        }
                    }
                ]
            }
        },
        "_source": ['document_id', 'question', 'answer']  # Fields to return in the results
    }

    # Perform the search with the constructed query
    es_results = es_client.search(index=index_name, body=search_query)

    # Extract and return the results
    result_docs = [hit["_source"] for hit in es_results["hits"]["hits"]]

    return result_docs

In [32]:
knn_results_combined_st = elastic_search_knn_combined_style(truth_question, embedding_type='SentenceTransformer', index_name='documents_st')
knn_combined_results_combined_st = [item['document_id'] for item in knn_results_combined_st]
knn_combined_results_combined_st, hit_rate_one(truth_document_id, knn_combined_results_combined_st), mrr_one(truth_document_id, knn_combined_results_combined_st)

(['doc_0_how_can_i_create_an_account_',
  'doc_16_can_i_order_without_creating_a',
  'doc_15_do_you_have_a_loyalty_program_',
  'doc_64_can_i_request_a_product_if_it_',
  'doc_1_what_payment_methods_do_you_ac',
  'doc_12_can_i_order_by_phone_',
  'doc_28_what_should_i_do_if_my_discoun',
  'doc_2_how_can_i_track_my_order_',
  'doc_11_what_is_your_price_matching_po',
  'doc_35_can_i_request_a_product_demons'],
 1,
 1.0)

In [33]:
knn_results_spacy = elastic_search_knn_combined_style(truth_question, embedding_type='Spacy', index_name='documents_spacy')
knn_combined_results_spacy = [item['document_id'] for item in knn_results_spacy]
knn_combined_results_spacy, hit_rate_one(truth_document_id, knn_combined_results_spacy), mrr_one(truth_document_id, knn_combined_results_spacy)

(['doc_40_can_i_request_a_product_that_i',
  'doc_70_can_i_request_a_product_that_i',
  'doc_23_can_i_order_a_product_that_is_',
  'doc_76_can_i_request_a_product_if_it_',
  'doc_73_can_i_request_a_product_that_i',
  'doc_64_can_i_request_a_product_if_it_',
  'doc_21_what_should_i_do_if_i_receive_',
  'doc_67_can_i_request_a_product_that_i',
  'doc_7_what_should_i_do_if_my_package',
  'doc_69_can_i_return_a_product_if_it_w'],
 0,
 0)

In [34]:
truth_question

'How do I sign up for an account on your website?'

In [ ]:
df_ground_truth_spacy

,question,document
0,How do I sign up for an account on your website?,doc_0_how_can_i_create_an_account_
1,Where can I find the Sign Up button to make a ...,doc_0_how_can_i_create_an_account_
2,What steps do I need to follow to register a n...,doc_0_how_can_i_create_an_account_
3,Can you tell me how to create an account on th...,doc_0_how_can_i_create_an_account_
4,How do I complete the registration process aft...,doc_0_how_can_i_create_an_account_
...,...,...
390,"If I bought something during a promo sale, can...",doc_78_can_i_return_a_product_if_it_w
391,Will I get refunded for the discounted price i...,doc_78_can_i_return_a_product_if_it_w
392,Are returns allowed for products purchased on ...,doc_78_can_i_return_a_product_if_it_w
393,"If I return an item I got with a discount, how...",doc_78_can_i_return_a_product_if_it_w


In [58]:
spacy_hit_rate_results_text = []
spacy_hit_rate_results_vector = []
spacy_hit_rate_results_vector_combined = []
spacy_mrr_results_text = []
spacy_mrr_results_vector = []
spacy_mrr_results_vector_combined = []

for index, row in tqdm(df_ground_truth_spacy.iterrows(), total=df_ground_truth_spacy.shape[0], desc="Processing rows"):
    document_id = row['document']
    question = row['question']
    
    text_results = elastic_search_text(question)
    text_results = [item['document_id'] for item in text_results]
    
    hit_rate_text = hit_rate_one(document_id, text_results)
    spacy_hit_rate_results_text.append(hit_rate_text)
    mrr_text = mrr_one(document_id, text_results)
    spacy_mrr_results_text.append(mrr_text)

    knn_results = elastic_search_knn(question, embedding_type='Spacy', index_name='documents_spacy')
    knn_results = [item['document_id'] for item in knn_results]

    hit_rate_vector = hit_rate_one(document_id, knn_results)
    spacy_hit_rate_results_vector.append(hit_rate_vector)
    mrr_vector = mrr_one(document_id, knn_results)
    spacy_mrr_results_vector.append(mrr_vector)

    knn_combined_results = elastic_search_knn_combined_style(question, embedding_type='Spacy', index_name='documents_spacy')
    knn_combined_results = [item['document_id'] for item in knn_combined_results]
    hit_rate_vector_combined = hit_rate_one(document_id, knn_combined_results)
    spacy_hit_rate_results_vector_combined.append(hit_rate_vector_combined)
    mrr_vector_combined = mrr_one(document_id, knn_combined_results)
    spacy_mrr_results_vector_combined.append(mrr_vector_combined)

Processing rows:   0%|          | 0/395 [00:00<?, ?it/s]

In [59]:
len(df_ground_truth_spacy), len(spacy_hit_rate_results_text), len(spacy_hit_rate_results_vector), len(spacy_mrr_results_text), len(spacy_mrr_results_vector), len(spacy_hit_rate_results_vector_combined), len(spacy_mrr_results_vector_combined)

(395, 395, 395, 395, 395, 395, 395)

In [62]:
df_ground_truth_spacy['hit_rate_text'] = spacy_hit_rate_results_text
df_ground_truth_spacy['hit_rate_vector'] = spacy_hit_rate_results_vector
df_ground_truth_spacy['hit_rate_vector_combined'] = spacy_hit_rate_results_vector_combined
df_ground_truth_spacy['mrr_text'] = spacy_mrr_results_text
df_ground_truth_spacy['mrr_vector'] = spacy_mrr_results_vector
df_ground_truth_spacy['mrr_vector_combined'] = spacy_mrr_results_vector_combined

In [63]:
df_ground_truth_spacy

,question,document,hit_rate_text,hit_rate_vector,hit_rate_vector_combined,mrr_text,mrr_vector,mrr_vector_combined
0,How do I sign up for an account on your website?,doc_0_how_can_i_create_an_account_,1,0,0,1.000000,0.000000,0.0
1,Where can I find the Sign Up button to make a ...,doc_0_how_can_i_create_an_account_,1,0,0,1.000000,0.000000,0.0
2,What steps do I need to follow to register a n...,doc_0_how_can_i_create_an_account_,1,0,0,0.200000,0.000000,0.0
3,Can you tell me how to create an account on th...,doc_0_how_can_i_create_an_account_,1,0,0,1.000000,0.000000,0.0
4,How do I complete the registration process aft...,doc_0_how_can_i_create_an_account_,1,0,0,0.500000,0.000000,0.0
...,...,...,...,...,...,...,...,...
390,"If I bought something during a promo sale, can...",doc_78_can_i_return_a_product_if_it_w,1,0,0,0.200000,0.000000,0.0
391,Will I get refunded for the discounted price i...,doc_78_can_i_return_a_product_if_it_w,1,1,1,0.500000,0.142857,0.2
392,Are returns allowed for products purchased on ...,doc_78_can_i_return_a_product_if_it_w,0,0,0,0.000000,0.000000,0.0
393,"If I return an item I got with a discount, how...",doc_78_can_i_return_a_product_if_it_w,0,0,0,0.000000,0.000000,0.0


In [64]:
df_ground_truth_spacy.describe()

,hit_rate_text,hit_rate_vector,hit_rate_vector_combined,mrr_text,mrr_vector,mrr_vector_combined
count,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
mean,0.870886,0.311392,0.298734,0.596194,0.110977,0.114676
std,0.335751,0.463650,0.458284,0.393669,0.231290,0.243161
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.250000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.500000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,0.133929,0.111111
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [65]:
df_ground_truth_spacy.to_csv('../data/ground_truth_metrics_retrieval-spacy.csv', index=False, sep=',', encoding='utf-8')

In [ ]:
st_hit_rate_results_text = []
st_hit_rate_results_vector = []
st_hit_rate_results_vector_combined = []
st_mrr_results_text = []
st_mrr_results_vector = []
st_mrr_results_vector_combined = []

for index, row in tqdm(df_ground_truth_st.iterrows(), total=df_ground_truth_st.shape[0], desc="Processing rows"):
    document_id = row['document']
    question = row['question']
    
    text_results = elastic_search_text(question)
    text_results = [item['document_id'] for item in text_results]
    
    hit_rate_text = hit_rate_one(document_id, text_results)
    st_hit_rate_results_text.append(hit_rate_text)
    mrr_text = mrr_one(document_id, text_results)
    st_mrr_results_text.append(mrr_text)

    knn_results = elastic_search_knn(question, embedding_type='SentenceTransformer', index_name='documents_st')
    knn_results = [item['document_id'] for item in knn_results]

    hit_rate_vector = hit_rate_one(document_id, knn_results)
    st_hit_rate_results_vector.append(hit_rate_vector)
    mrr_vector = mrr_one(document_id, knn_results)
    st_mrr_results_vector.append(mrr_vector)

    knn_combined_results = elastic_search_knn_combined_style(question, embedding_type='SentenceTransformer', index_name='documents_st')
    knn_combined_results = [item['document_id'] for item in knn_combined_results]
    hit_rate_vector_combined = hit_rate_one(document_id, knn_combined_results)
    st_hit_rate_results_vector_combined.append(hit_rate_vector_combined)
    mrr_vector_combined = mrr_one(document_id, knn_combined_results)
    st_mrr_results_vector_combined.append(mrr_vector_combined)

Processing rows:   0%|          | 0/395 [00:00<?, ?it/s]

In [ ]:
len(df_ground_truth_st), len(st_hit_rate_results_text), len(st_hit_rate_results_vector), len(st_mrr_results_text), len(st_mrr_results_vector), len(st_hit_rate_results_vector_combined), len(st_mrr_results_vector_combined)

(395, 395, 395, 395, 395, 395, 395)

In [46]:
df_ground_truth_st['hit_rate_text'] = st_hit_rate_results_text
df_ground_truth_st['hit_rate_vector'] = st_hit_rate_results_vector
df_ground_truth_st['hit_rate_vector_combined'] = st_hit_rate_results_vector_combined
df_ground_truth_st['mrr_text'] = st_mrr_results_text
df_ground_truth_st['mrr_vector'] = st_mrr_results_vector
df_ground_truth_st['mrr_vector_combined'] = st_mrr_results_vector_combined

In [47]:
df_ground_truth_st

,question,document,hit_rate_text,hit_rate_vector,hit_rate_vector_combined,mrr_text,mrr_vector,mrr_vector_combined
0,How do I sign up for an account on your website?,doc_0_how_can_i_create_an_account_,1,1,1,1.000000,1.00,1.00
1,Where can I find the Sign Up button to make a ...,doc_0_how_can_i_create_an_account_,1,1,1,1.000000,1.00,1.00
2,What steps do I need to follow to register a n...,doc_0_how_can_i_create_an_account_,1,1,1,0.200000,1.00,1.00
3,Can you tell me how to create an account on th...,doc_0_how_can_i_create_an_account_,1,1,1,1.000000,1.00,1.00
4,How do I complete the registration process aft...,doc_0_how_can_i_create_an_account_,1,1,1,0.500000,1.00,1.00
...,...,...,...,...,...,...,...,...
390,"If I bought something during a promo sale, can...",doc_78_can_i_return_a_product_if_it_w,1,1,1,0.200000,1.00,1.00
391,Will I get refunded for the discounted price i...,doc_78_can_i_return_a_product_if_it_w,1,1,1,0.500000,1.00,1.00
392,Are returns allowed for products purchased on ...,doc_78_can_i_return_a_product_if_it_w,0,1,1,0.000000,1.00,1.00
393,"If I return an item I got with a discount, how...",doc_78_can_i_return_a_product_if_it_w,0,1,1,0.000000,0.25,0.25


In [48]:
df_ground_truth_st.describe()

,hit_rate_text,hit_rate_vector,hit_rate_vector_combined,mrr_text,mrr_vector,mrr_vector_combined
count,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
mean,0.870886,0.994937,0.994937,0.596194,0.865445,0.865572
std,0.335751,0.071066,0.071066,0.393669,0.267970,0.267469
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,1.000000,0.250000,1.000000,1.000000
50%,1.000000,1.000000,1.000000,0.500000,1.000000,1.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [50]:
df_ground_truth_st.to_csv('../data/ground_truth_metrics_retrieval-sentence_transformer.csv', index=False, sep=',', encoding='utf-8')